In [1]:
import polars as pl
from constant import openalex_path

In [ ]:
lpu_lf = pl.scan_parquet(openalex_path + "/works/primary_location_landing_page_url/*")
lpu_lf.head().collect(engine="streaming")

In [4]:
arxiv_candidates = (
    lpu_lf
    .filter(
        pl.col("primary_location_landing_page_url").str.to_lowercase().str.contains("/abs/")
    )
    .collect(engine="streaming")
)

In [2]:
preprint_opal = pl.read_parquet("./data/preprints_opal.parquet")

In [3]:
preprint_opal.filter(pl.col("publication_year")<2022).count()

doi,primary_category,server,curated_doi,published_date,work_id,primary_location_id,publication_year
u32,u32,u32,u32,u32,u32,u32,u32
1056343,1056341,1056343,142657,1056343,1056343,991926,1056343


In [4]:
arxiv_missing_by_category = (
    preprint_opal
    .filter(pl.col("server") == "arxiv")
    .with_columns(
        pl.col("work_id").is_null().alias("is_missing")
    )
    .with_columns(
        pl.col("is_missing").sum().over("primary_category").alias("missing_count"),
        pl.col("doi").count().over("primary_category").alias("total_count"),
    )
    .select("primary_category", "total_count", "missing_count")
    .unique("primary_category", keep="first")
    .with_columns(
        (pl.col("missing_count") / pl.col("total_count") * 100).alias("missing_percentage")
    )
)

In [5]:
arxiv_missing_by_category.sort(pl.col("missing_percentage")).write_csv("./data/arxiv_missing_by_category_0113.csv")

In [6]:
arxiv_missing = (
    preprint_opal
    .filter(
        pl.col("server") == "arxiv",
        pl.col("work_id").is_null()
    )
)

In [35]:
(
    preprint_opal
    .filter(
        pl.col("server") == "arxiv",
        pl.col("work_id").is_not_null()
    )
)

doi,primary_category,server,curated_doi,published_date,work_id,primary_location_id,publication_year
str,str,str,str,datetime[μs],str,str,i64
"""10.48550/arxiv.2409.20153""","""hep-ph""","""arxiv""",null,2024-09-30 10:00:57,"""W4403407026""","""S4306400194""",2024
"""10.48550/arxiv.2410.10397""","""cs.LG""","""arxiv""",null,2024-10-14 11:37:36,"""W4403407106""","""S4306400194""",2024
"""10.48550/arxiv.2408.12563""","""astro-ph.EP""","""arxiv""",null,2024-08-22 17:25:59,"""W4403555258""","""S4306400194""",2024
"""10.48550/arxiv.2409.02925""","""math.GM""","""arxiv""",null,2024-08-13 02:21:39,"""W4403555419""","""S4306400194""",2024
"""10.48550/arxiv.2409.02947""","""math.GM""","""arxiv""",null,2024-08-29 07:05:43,"""W4403555499""","""S4306400194""",2024
…,…,…,…,…,…,…,…
"""10.48550/arxiv.0904.2940""","""math.FA""","""arxiv""",null,2009-04-19 00:00:00,"""W2262339942""","""S4306400194""",2009
"""10.48550/arxiv.1706.05457""","""math.SP""","""arxiv""",null,2017-06-16 00:00:00,"""W2675172783""","""S4306400194""",2017
"""10.48550/arxiv.1811.05353""","""math.NA""","""arxiv""",null,2018-11-13 00:00:00,"""W4289293270""","""S4306400194""",2018


In [9]:
arxiv_missing

doi,primary_category,server,curated_doi,published_date,work_id,primary_location_id,publication_year
str,str,str,str,datetime[μs],str,str,i64
"""10.48550/arxiv.2206.03275""","""cs.CY""","""arxiv""","""10.1145/3531146.3533186""",2022-06-03 15:44:44,null,null,null
"""10.48550/arxiv.1702.07945""","""cs.IT""","""arxiv""","""10.1109/TSP.2018.2835403""",2017-02-25 00:00:00,null,null,null
"""10.48550/arxiv.0908.0647""","""cond-mat.mes-hall""","""arxiv""","""10.1103/PhysRevB.94.155429""",2009-08-05 00:00:00,null,null,null
"""10.48550/arxiv.1411.4330""","""hep-th""","""arxiv""","""10.1103/PhysRevD.90.115028""",2014-11-16 00:00:00,null,null,null
"""10.48550/arxiv.1610.06517""","""math.PR""","""arxiv""","""10.1007/s00220-018-3201-1""",2016-10-20 00:00:00,null,null,null
…,…,…,…,…,…,…,…
"""10.48550/arxiv.1607.08056""","""cond-mat.mes-hall""","""arxiv""","""10.1103/PhysRevB.93.245420""",2016-07-27 00:00:00,null,null,null
"""10.48550/arxiv.1705.01160""","""math.CA""","""arxiv""","""10.1007/s10569-018-9837-5""",2017-05-02 00:00:00,null,null,null
"""10.48550/arxiv.1507.07447""","""math.DG""","""arxiv""","""10.1007/s00023-015-0432-y""",2015-07-27 00:00:00,null,null,null


In [ ]:
import requests
url = "https://api.openalex.org/works?filter=publication_year:2005,primary_location.source.id:s4306400194&per_page=10"
response = requests.get(url)
data = response.json()


In [57]:
import requests
arxiv_id = "10.48550/arxiv.1411.4330"
url = f"https://api.openalex.org/works?filter=doi_starts_with:{arxiv_id}&per_page=10&include_xpac=true"
response = requests.get(url)
data = response.json()

In [53]:
arxiv_ids = arxiv_missing.select("doi").to_series().to_list()
arxiv_ids[:3], len(arxiv_ids)

(['10.48550/arxiv.2206.03275',
  '10.48550/arxiv.1702.07945',
  '10.48550/arxiv.0908.0647'],
 993199)

In [64]:
import json
import requests
from pathlib import Path
import glob

# ---------- 設定 ----------
CHUNK_SIZE = 100
OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)

BASE_URL = "https://api.openalex.org/works"

# ---------- 既取得IDの復元 ----------
def load_already_fetched_ids(out_dir: Path) -> set[str]:
    fetched = set()
    for path in glob.glob(str(out_dir / "works_*.jsonl")):
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                # OpenAlexの標準的な DOI 格納場所
                doi = obj.get("doi")
                if doi:
                    fetched.add(doi.lower().removeprefix("https://doi.org/"))
    return fetched

already_fetched = load_already_fetched_ids(OUT_DIR)

print(f"Already fetched: {len(already_fetched)}")

# ---------- バッファ ----------
buffer = []
file_idx = (
    max(
        [int(p.stem.split("_")[1]) for p in OUT_DIR.glob("works_*.jsonl")],
        default=0,
    )
    + 1
)

def flush_buffer(buf, idx):
    path = OUT_DIR / f"works_{idx:04d}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for record in buf:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

# ---------- メインループ ----------
for arxiv_id in arxiv_ids:
    arxiv_id = arxiv_id.lower()

    if arxiv_id in already_fetched:
        continue  # ← ここが肝

    url = (
        f"{BASE_URL}"
        f"?filter=doi_starts_with:{arxiv_id}"
        "&per_page=1"
        "&include_xpac=true"
    )

    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    for work in data.get("results", []):
        buffer.append(work)
        doi = work.get("doi")
        if doi:
            already_fetched.add(doi.lower())

        if len(buffer) == CHUNK_SIZE:
            flush_buffer(buffer, file_idx)
            buffer.clear()
            file_idx += 1

# ---------- 端数 ----------
if buffer:
    flush_buffer(buffer, file_idx)

Already fetched: 100


KeyboardInterrupt: 

In [65]:
arxiv_missing.write_ndjson("./data/arxiv_missing_preprints.jsonl")

In [25]:
arxiv_missing.group_by(pl.col("published_date").dt.year()).len().sort("published_date")

published_date,len
i32,u32
1993,4720
1994,6710
1995,8542
1996,10314
1997,12929
…,…
2020,36630
2021,34418
2022,18898


In [77]:
df_ = (
    pl.scan_ndjson("data-harvesting/out/*")
    .select(["id","doi","referenced_works", "cited_by_count"])
    .with_columns(
        pl.col("id").str.to_lowercase().str.replace("https://openalex.org/", ""),
        pl.col("doi").str.to_lowercase().str.replace("https://doi.org/", "")
    )
)
df_.head().collect(engine="streaming")

id,doi,referenced_works,cited_by_count
str,str,list[str],i64
"""w6891988964""","""10.48550/arxiv.2206.03275""",[],1
"""w6891763081""","""10.48550/arxiv.1702.07945""",[],0
"""w2963511202""","""10.48550/arxiv.0908.0647""","[""https://openalex.org/W2071246696""]",0
"""w6891818973""","""10.48550/arxiv.1411.4330""",[],0
"""w6967419135""","""10.48550/arxiv.1610.06517""",[],0


In [80]:
ref_lf = pl.scan_parquet(openalex_path + "/works/referenced_works/*")
ref_lf.explode("referenced_works").filter(pl.col("referenced_works") == "W3098176052").collect(engine="streaming")

work_id,referenced_works
str,str
"""W2784914100""","""W3098176052"""
"""W2587219671""","""W3098176052"""
"""W2531421330""","""W3098176052"""
"""W2341596688""","""W3098176052"""
"""W1977720663""","""W3098176052"""
…,…
"""W2074129889""","""W3098176052"""
"""W2117754529""","""W3098176052"""
"""W2130229375""","""W3098176052"""


In [79]:
df_.filter(pl.col("cited_by_count")>0).sort("cited_by_count", descending=True).head(30).collect(engine="streaming")

id,doi,referenced_works,cited_by_count
str,str,list[str],i64
"""w3098176052""","""10.48550/arxiv.cond-mat/040248…",[],179
"""w1805300852""","""10.48550/arxiv.0803.0004""","[""https://openalex.org/W2020185953"", ""https://openalex.org/W2108703509"", … ""https://openalex.org/W2038712212""]",67
"""w3100589190""","""10.48550/arxiv.astro-ph/040903…",[],60
"""w2603312341""","""10.48550/arxiv.1703.05703""","[""https://openalex.org/W3105524070"", ""https://openalex.org/W2211090155"", … ""https://openalex.org/W2159449603""]",53
"""w2075327077""","""10.1137/130912219""","[""https://openalex.org/W2070514473"", ""https://openalex.org/W2105640560"", … ""https://openalex.org/W1540059687""]",45
…,…,…,…
"""w6967484631""","""10.48550/arxiv.1912.04424""",[],8
"""w3025980542""","""10.48550/arxiv.cond-mat/011006…",[],7
"""w2043621486""","""10.48550/arxiv.1004.4472""","[""https://openalex.org/W2081811158"", ""https://openalex.org/W2093307998"", … ""https://openalex.org/W1571745730""]",7


In [ ]:
arxiv_candidate = source_lf.filter(
    pl.col("primary_location_id")=="S4306400194"
).collect(engine="streaming")

In [ ]:
arxiv_candidate.height

In [ ]:
arxiv_candidate.lazy().join(preprints_opal.filter(pl.col("server")=="arxiv"), on="work_id", how="semi").collect(engine="streaming")